# Append calendar-aging day to lifetime diagnostic summaries

Run this notebook after `A01_cell_lifetime_diagnostics.ipynb` and before the Figure 6 group-processing notebook. It appends the calendar-aging exposure time (`day`) to each `cell*_voltage_fit_summary.csv` using the raw lifetime data. Cycling cells keep `day = NaN`; calendar-aging cells use the voltage/temperature rules listed below.

The notebook reads raw lifetime data from the repository-local `data/cell_lifetime_data` folder. If a local data copy is intentionally incomplete, rows without available raw data preserve the existing day information and are reported for review.


In [ ]:
from pathlib import Path
import gzip
import pickle
import re

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## 1. Paths and user settings


In [ ]:
def find_repo_root(start: Path) -> Path:
    """Locate the repository root from this notebook path."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "F6_add_calendar_day.ipynb").exists():
    NOTEBOOK_DIR = Path("code/plotting/f6").resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

BATCH_RESULTS_ROOT = REPO_ROOT / "code" / "diagnostic_algorithm_lifetime_crate" / "batch_results" / "A01_cell_lifetime_diagnostics"
CSV_GLOB = "cell*/cell*_voltage_fit_summary.csv"

# Raw lifetime data are expected to live inside the repository. The public
# Raw lifetime data are expected in the repository-local data folder.
# Incomplete local data copies are handled without overwriting existing day values.
RAW_DATA_ROOT = REPO_ROOT / "data" / "cell_lifetime_data"

WRITE_BACK = True
MASTER_OUT = BATCH_RESULTS_ROOT / "Master_with_calendar_day.csv"

print("REPO_ROOT =", REPO_ROOT)
print("BATCH_RESULTS_ROOT =", BATCH_RESULTS_ROOT)
print("RAW_DATA_ROOT =", RAW_DATA_ROOT)
print("WRITE_BACK =", WRITE_BACK)


## 2. Cell schema and calendar-aging rules


In [ ]:
TYPE1_CELLS = {
    1, 8, 9, 16, 18, 22, 23, 24, 26, 28, 30, 31, 32, 35, 40, 41, 43, 44, 49, 51, 53, 54, 61, 65, 66, 75,
    45, 4, 57, 36, 52, 3, 42, 48, 5, 46,
}
TYPE2_CELLS = {
    2, 7, 12, 16, 17, 25, 27, 32, 47, 49, 56, 60, 64, 71, 72, 74, 11, 55, 19,
}
PREFER_OVERLAP = "type2"

# Calendar-aging day is counted only during the storage condition associated
# with each cell. A value of None means no temperature threshold is applied.
CELL_CONDITIONS = {
    45: (40, 3.2),
    4:  (40, 3.6),
    57: (40, 3.6),
    36: (40, 3.8),
    52: (40, 3.8),
    3:  (50, 3.6),
    42: (50, 3.4),
    48: (50, 3.4),
    5:  (None, 4.1),
    46: (None, 4.1),
}
CALENDAR_CELLS = set(CELL_CONDITIONS)


## 3. Raw-data loading helpers


In [ ]:
def infer_cell_from_path(path_str: str):
    match = re.search(r"cell(\d{3})", str(path_str), flags=re.IGNORECASE)
    return int(match.group(1)) if match else np.nan


def schema_id_for_cell(cell: int):
    in_type1 = cell in TYPE1_CELLS
    in_type2 = cell in TYPE2_CELLS
    if in_type1 and not in_type2:
        return "type1"
    if in_type2 and not in_type1:
        return "type2"
    if in_type1 and in_type2:
        return PREFER_OVERLAP
    return None


def unwrap_df(obj):
    if isinstance(obj, pd.DataFrame):
        return obj
    if isinstance(obj, dict):
        for key in ("data", "df", "CD", "cd", "table"):
            value = obj.get(key)
            if isinstance(value, pd.DataFrame):
                return value
        for value in obj.values():
            if isinstance(value, pd.DataFrame):
                return value
    raise TypeError(f"Expected a DataFrame or a dict containing one, got {type(obj)}")


def read_pkl_gz(path: Path):
    with gzip.open(path, "rb") as handle:
        return pickle.load(handle)


def candidate_raw_roots(raw_root: Path, schema: str):
    roots = [raw_root]
    if raw_root.name != "cell_lifetime_data":
        roots.append(raw_root / "cell_lifetime_data")

    expanded = []
    for root in roots:
        if schema == "type1":
            expanded.extend([root / "type1", root / "PROCESSED" / "GMFEB23S", root / "GMFEB23S", root])
        else:
            expanded.extend([root / "type2", root / "Processed" / "GMFEB23S", root / "GMFEB23S", root])

    unique = []
    seen = set()
    for root in expanded:
        key = str(root)
        if key not in seen:
            unique.append(root)
            seen.add(key)
    return unique


def resolve_cell_dir(root: Path, cell: int):
    cell_no = f"{cell:03d}"
    candidates = [
        root / f"GMFEB23S_CELL{cell_no}",
        root / f"GMFEB23s_CELL{cell_no}",
        root / f"CELL{cell_no}",
        root / cell_no,
    ]
    for directory in candidates:
        if directory.is_dir():
            return directory
    if root.is_dir():
        hits = [p for p in root.glob(f"*{cell_no}*") if p.is_dir()]
        if len(hits) == 1:
            return hits[0]
    return None


def load_raw_cd(cell: int):
    schema = schema_id_for_cell(cell)
    if schema is None:
        raise ValueError(f"Cell {cell:03d} has unknown raw-data schema.")

    filenames = ["CD.pkl.gz"] if schema == "type1" else ["cell_data.pkl.gz", "CD.pkl.gz"]
    for root in candidate_raw_roots(RAW_DATA_ROOT, schema):
        cell_dir = resolve_cell_dir(root, cell)
        if cell_dir is None:
            continue
        for filename in filenames:
            cd_path = cell_dir / filename
            if cd_path.is_file():
                return unwrap_df(read_pkl_gz(cd_path)), schema, cd_path

    raise FileNotFoundError(f"raw CD data unavailable for cell {cell:03d}")


## 4. Calendar-day calculation


In [ ]:
def get_raw_columns(cd_df: pd.DataFrame, schema: str):
    """Return schema-specific raw-data column names used in the day calculation."""
    if schema == "type1":
        return "Ah throughput [A.h]", "Voltage [V]", "Temperature [degC]", "Time [ms]"

    ah_col = "capacity(ah)"
    volt_col = "voltage(v)"
    time_col = "timestamp"
    lower_map = {str(col).lower(): col for col in cd_df.columns}
    temp_col = None
    for candidate in ["temperature(degc)", "temperature [degc]", "temperature [degC]", "temperature"]:
        if candidate.lower() in lower_map:
            temp_col = lower_map[candidate.lower()]
            break
    return ah_col, volt_col, temp_col, time_col


def time_to_milliseconds(values, schema: str):
    if schema == "type2":
        timestamps = pd.to_datetime(values, errors="coerce", utc=True)
        out = pd.Series(timestamps.astype("int64") / 1e6, index=values.index, dtype="float64")
        out[timestamps.isna()] = np.nan
        return out
    return pd.to_numeric(values, errors="coerce")


def add_calendar_day_to_summary(df_summary: pd.DataFrame, cell: int):
    """Append calendar-aging day to one cell summary table."""
    df = df_summary.copy()
    df["is_calendar"] = cell in CALENDAR_CELLS
    df["calendar_day_status"] = "not_calendar"

    if cell not in CALENDAR_CELLS:
        df["day"] = np.nan
        return df

    try:
        cd_df, schema, cd_path = load_raw_cd(cell)
    except Exception as exc:
        if "day" not in df.columns:
            df["day"] = np.nan
        df["calendar_day_status"] = "raw_missing_day_preserved"
        return df

    temp_threshold, volt_threshold = CELL_CONDITIONS[cell]
    ah_col, volt_col, temp_col, time_col = get_raw_columns(cd_df, schema)
    required = [ah_col, volt_col, time_col]
    missing = [col for col in required if col not in cd_df.columns]
    if missing:
        if "day" not in df.columns:
            df["day"] = np.nan
        df["calendar_day_status"] = "missing_raw_columns_day_preserved"
        return df

    ah_values = pd.to_numeric(cd_df[ah_col], errors="coerce")
    volt_values = pd.to_numeric(cd_df[volt_col], errors="coerce")
    time_values = time_to_milliseconds(cd_df[time_col], schema)

    if temp_col is not None and temp_col in cd_df.columns:
        temp_values = pd.to_numeric(cd_df[temp_col], errors="coerce")
    else:
        temp_values = pd.Series(np.nan, index=cd_df.index)

    days = []
    for ah_throughput in pd.to_numeric(df["Ah_throughput"], errors="coerce"):
        condition = (volt_values > volt_threshold) & (ah_values <= ah_throughput)
        if temp_threshold is not None:
            condition &= temp_values > temp_threshold

        t_section = time_values[condition]
        if np.isfinite(t_section).sum() >= 2:
            days.append(float(np.nanmax(t_section) - np.nanmin(t_section)) / (1000 * 60 * 60 * 24))
        else:
            days.append(0.0)

    df["day"] = days
    try:
        cd_label = str(cd_path.relative_to(REPO_ROOT))
    except ValueError:
        cd_label = cd_path.name
    df["calendar_day_status"] = f"success: {cd_label}"
    return df


## 5. Apply to all lifetime summary CSVs


In [ ]:
csv_paths = sorted(BATCH_RESULTS_ROOT.glob(CSV_GLOB))
print(f"Found {len(csv_paths)} lifetime summary CSVs")

all_tables = []
for summary_path in csv_paths:
    cell = infer_cell_from_path(str(summary_path))
    if not np.isfinite(cell):
        print(f"Skip {summary_path.name}: cannot infer cell number")
        continue
    cell = int(cell)

    df = pd.read_csv(summary_path)
    if "Ah_throughput" not in df.columns:
        print(f"Skip {summary_path.name}: missing Ah_throughput")
        continue

    df_day = add_calendar_day_to_summary(df, cell)
    df_day["cell"] = cell
    try:
        df_day["source_file"] = str(summary_path.relative_to(REPO_ROOT))
    except ValueError:
        df_day["source_file"] = summary_path.name

    all_tables.append(df_day)
    if WRITE_BACK:
        df_day.to_csv(summary_path, index=False)

    status = str(df_day["calendar_day_status"].iloc[0]) if "calendar_day_status" in df_day.columns else "unknown"
    print(f"Done cell {cell:03d}: {status.split(':', 1)[0]}")

Master_with_day = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
print("Master shape:", Master_with_day.shape)
Master_with_day.head()


## 6. Save merged table and quick checks


In [ ]:
Master_with_day.to_csv(MASTER_OUT, index=False)
print("Saved:", MASTER_OUT)

if not Master_with_day.empty:
    check_cols = [col for col in ["cell", "rpt_seq", "Ah_throughput", "day", "is_calendar", "calendar_day_status"] if col in Master_with_day.columns]
    display(Master_with_day.loc[Master_with_day["is_calendar"], check_cols].sort_values(["cell", "Ah_throughput"]).head(50))

csv_cells = sorted({int(infer_cell_from_path(str(path))) for path in csv_paths if np.isfinite(infer_cell_from_path(str(path)))})
print("Calendar cells found in batch_results:", sorted(set(csv_cells) & CALENDAR_CELLS))
print("Calendar status counts:")
print(Master_with_day.get("calendar_day_status", pd.Series(dtype=object)).value_counts().head(20))
